In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_parquet('data_sample/news_prices_full_processed.parquet')
print(df.shape)
# lose not so much data so let's drop rows with missing price data to have the cleaner dataset for the target creation
df = df.loc[(df['prev_day_price'].notna()) & (df['curr_day_price'].notna()) & (df['next_day_price'].notna())]
print(df.shape)
df

(100428, 27)
(87995, 27)


,date,prev_date,future_date,title,description,maintext,ticker,prev_day_price,curr_day_price,next_day_price,...,emotion_fear,emotion_joy,emotion_sadness,emotion_disgust,emotion_surprise,emotion_neutral,date_day_of_week,prev_day_of_week,future_day_of_week,language
2,2019-01-22,2019-01-21,2019-01-23,UBS Warns on Client Activity After $13 Billion...,Withdrawals at the Zurich-based bank’s key glo...,(Bloomberg) -- UBS Group AG warned client acti...,C,63.12000,61.85000,62.13000,...,0.037095,0.002821,0.799185,0.043700,0.022429,0.071112,Tuesday,Monday,Wednesday,en
3,2019-08-01,2019-07-31,2019-08-02,Shopify Boosts Outlook on New Online Offerings,(Bloomberg) -- Shopify Inc. shares continued t...,(Bloomberg) -- Shopify Inc. shares continued t...,SHOP,317.88000,341.39001,332.19000,...,0.007578,0.095697,0.013748,0.009044,0.107670,0.752265,Thursday,Wednesday,Friday,en
4,2019-10-31,2019-10-30,2019-11-01,Wayfair Plunges as Forecast Miss Heightens Gro...,(Bloomberg) -- Wayfair Inc. plunged near its l...,(Bloomberg) -- Wayfair Inc. plunged near its l...,C,72.97000,71.86000,73.84000,...,0.444956,0.007151,0.185429,0.010559,0.080362,0.254296,Thursday,Wednesday,Friday,en
5,2019-02-08,2019-02-07,2019-02-09,Skyscrapers Made of Wood Are Making a Comeback,"Sidewalk Labs LLC, a unit of Google parent Alp...",(Bloomberg) -- More than a century after steel...,GOOGL,1098.70996,1095.06006,1095.01001,...,0.001772,0.026602,0.002590,0.008262,0.009297,0.944932,Friday,Thursday,Saturday,en
6,2019-01-24,2019-01-23,2019-01-25,Intel Sales Miss as Data-Center Demand Slows; ...,Revenue in the current period will be about $1...,(Bloomberg) -- Intel Corp. reported lower-than...,GOOGL,1075.56995,1073.90002,1090.98999,...,0.515582,0.006684,0.080193,0.018094,0.074352,0.293145,Thursday,Wednesday,Friday,en
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100423,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt alien...,You can ride giant creatures and companions ca...,Benzinga\nWarren Buffett's Berkshire Cuts Appl...,AAPL,133.19000,130.84000,129.71001,...,0.025531,0.007261,0.062263,0.011118,0.510062,0.360592,Wednesday,Tuesday,Thursday,en
100424,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt alien...,You can ride giant creatures and companions ca...,Benzinga\nWarren Buffett's Berkshire Cuts Appl...,CVX,93.13000,95.92000,95.00000,...,0.025531,0.007261,0.062263,0.011118,0.510062,0.360592,Wednesday,Tuesday,Thursday,en
100425,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt alien...,You can ride giant creatures and companions ca...,Benzinga\nWarren Buffett's Berkshire Cuts Appl...,MRK,74.25000,75.54000,75.41000,...,0.025531,0.007261,0.062263,0.011118,0.510062,0.360592,Wednesday,Tuesday,Thursday,en
100426,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt alien...,You can ride giant creatures and companions ca...,Benzinga\nWarren Buffett's Berkshire Cuts Appl...,BRK,245.28000,370500.00000,245.25000,...,0.025531,0.007261,0.062263,0.011118,0.510062,0.360592,Wednesday,Tuesday,Thursday,en


### Percentage changes of prices (target creation)

In [3]:
# price-return features
df['curr_prev_per_change'] = (df['curr_day_price'] - df['prev_day_price']) / df['prev_day_price']
df['next_curr_per_change'] = (df['next_day_price'] - df['curr_day_price']) / df['curr_day_price'] # THIS IS OUR TARGET

# 1 if next-day up, 0 otherwise
df['curr_prev_pos_change'] = (df['curr_prev_per_change'] > 0).astype(int)
df['next_curr_pos_change'] = (df['next_curr_per_change'] > 0).astype(int)

In [4]:
df.head(3)

,date,prev_date,future_date,title,description,maintext,ticker,prev_day_price,curr_day_price,next_day_price,...,emotion_surprise,emotion_neutral,date_day_of_week,prev_day_of_week,future_day_of_week,language,curr_prev_per_change,next_curr_per_change,curr_prev_pos_change,next_curr_pos_change
2,2019-01-22,2019-01-21,2019-01-23,UBS Warns on Client Activity After $13 Billion...,Withdrawals at the Zurich-based bank’s key glo...,(Bloomberg) -- UBS Group AG warned client acti...,C,63.12,61.85000,62.13,...,0.022429,0.071112,Tuesday,Monday,Wednesday,en,-0.020120,0.004527,0,1
3,2019-08-01,2019-07-31,2019-08-02,Shopify Boosts Outlook on New Online Offerings,(Bloomberg) -- Shopify Inc. shares continued t...,(Bloomberg) -- Shopify Inc. shares continued t...,SHOP,317.88,341.39001,332.19,...,0.107670,0.752265,Thursday,Wednesday,Friday,en,0.073959,-0.026949,1,0
4,2019-10-31,2019-10-30,2019-11-01,Wayfair Plunges as Forecast Miss Heightens Gro...,(Bloomberg) -- Wayfair Inc. plunged near its l...,(Bloomberg) -- Wayfair Inc. plunged near its l...,C,72.97,71.86000,73.84,...,0.080362,0.254296,Thursday,Wednesday,Friday,en,-0.015212,0.027554,0,1


### Feature generation

In [5]:
# text-length / style signals
df['title_word_count'] = (df['title'].astype(str).str.split(' ').apply(lambda x: len([w for w in x if w.strip() != ""])))
df['description_word_count'] = (df['description'].astype(str).str.split(' ').apply(lambda x: len([w for w in x if w.strip() != ""])))
df['maintext_word_count'] = (df['maintext'].astype(str).str.split(' ').apply(lambda x: len([w for w in x if w.strip() != ""])))

### Combine texts

In [6]:
df['title + description + maintext'] = df['title'].astype(str) + ' ' + df['description'].astype(str) + ' ' + df['maintext'].astype(str)
df['title + description'] = df['title'].astype(str) + ' ' + df['description'].astype(str)

In [7]:
df

,date,prev_date,future_date,title,description,maintext,ticker,prev_day_price,curr_day_price,next_day_price,...,language,curr_prev_per_change,next_curr_per_change,curr_prev_pos_change,next_curr_pos_change,title_word_count,description_word_count,maintext_word_count,title + description + maintext,title + description
2,2019-01-22,2019-01-21,2019-01-23,UBS Warns on Client Activity After $13 Billion...,Withdrawals at the Zurich-based bank’s key glo...,(Bloomberg) -- UBS Group AG warned client acti...,C,63.12000,61.85000,62.13000,...,en,-0.020120,0.004527,0,1,10,46,707,UBS Warns on Client Activity After $13 Billion...,UBS Warns on Client Activity After $13 Billion...
3,2019-08-01,2019-07-31,2019-08-02,Shopify Boosts Outlook on New Online Offerings,(Bloomberg) -- Shopify Inc. shares continued t...,(Bloomberg) -- Shopify Inc. shares continued t...,SHOP,317.88000,341.39001,332.19000,...,en,0.073959,-0.026949,1,0,7,42,459,Shopify Boosts Outlook on New Online Offerings...,Shopify Boosts Outlook on New Online Offerings...
4,2019-10-31,2019-10-30,2019-11-01,Wayfair Plunges as Forecast Miss Heightens Gro...,(Bloomberg) -- Wayfair Inc. plunged near its l...,(Bloomberg) -- Wayfair Inc. plunged near its l...,C,72.97000,71.86000,73.84000,...,en,-0.015212,0.027554,0,1,8,51,315,Wayfair Plunges as Forecast Miss Heightens Gro...,Wayfair Plunges as Forecast Miss Heightens Gro...
5,2019-02-08,2019-02-07,2019-02-09,Skyscrapers Made of Wood Are Making a Comeback,"Sidewalk Labs LLC, a unit of Google parent Alp...",(Bloomberg) -- More than a century after steel...,GOOGL,1098.70996,1095.06006,1095.01001,...,en,-0.003322,-0.000046,0,0,8,48,657,Skyscrapers Made of Wood Are Making a Comeback...,Skyscrapers Made of Wood Are Making a Comeback...
6,2019-01-24,2019-01-23,2019-01-25,Intel Sales Miss as Data-Center Demand Slows; ...,Revenue in the current period will be about $1...,(Bloomberg) -- Intel Corp. reported lower-than...,GOOGL,1075.56995,1073.90002,1090.98999,...,en,-0.001553,0.015914,0,1,9,54,675,Intel Sales Miss as Data-Center Demand Slows; ...,Intel Sales Miss as Data-Center Demand Slows; ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100423,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt alien...,You can ride giant creatures and companions ca...,Benzinga\nWarren Buffett's Berkshire Cuts Appl...,AAPL,133.19000,130.84000,129.71001,...,en,-0.017644,-0.008636,0,0,10,13,2821,'No Man's Sky' update lets players adopt alien...,'No Man's Sky' update lets players adopt alien...
100424,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt alien...,You can ride giant creatures and companions ca...,Benzinga\nWarren Buffett's Berkshire Cuts Appl...,CVX,93.13000,95.92000,95.00000,...,en,0.029958,-0.009591,1,0,10,13,2821,'No Man's Sky' update lets players adopt alien...,'No Man's Sky' update lets players adopt alien...
100425,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt alien...,You can ride giant creatures and companions ca...,Benzinga\nWarren Buffett's Berkshire Cuts Appl...,MRK,74.25000,75.54000,75.41000,...,en,0.017374,-0.001721,1,0,10,13,2821,'No Man's Sky' update lets players adopt alien...,'No Man's Sky' update lets players adopt alien...
100426,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt alien...,You can ride giant creatures and companions ca...,Benzinga\nWarren Buffett's Berkshire Cuts Appl...,BRK,245.28000,370500.00000,245.25000,...,en,1509.518591,-0.999338,1,0,10,13,2821,'No Man's Sky' update lets players adopt alien...,'No Man's Sky' update lets players adopt alien...


In [8]:
df.columns

Index(['date', 'prev_date', 'future_date', 'title', 'description', 'maintext',
       'ticker', 'prev_day_price', 'curr_day_price', 'next_day_price',
       'named_entities', 'mentioned_companies', 'related_companies',
       'sentiment_negative', 'sentiment_neutral', 'sentiment_positive',
       'emotion_anger', 'emotion_fear', 'emotion_joy', 'emotion_sadness',
       'emotion_disgust', 'emotion_surprise', 'emotion_neutral',
       'date_day_of_week', 'prev_day_of_week', 'future_day_of_week',
       'language', 'curr_prev_per_change', 'next_curr_per_change',
       'curr_prev_pos_change', 'next_curr_pos_change', 'title_word_count',
       'description_word_count', 'maintext_word_count',
       'title + description + maintext', 'title + description'],
      dtype='object')

### Read additional data

In [9]:
final_df = pd.read_csv('data_sample/prices_for_merging.csv')
final_df['Date'] = pd.to_datetime(final_df['Date'], errors='coerce', infer_datetime_format=True)
final_df['date'] = final_df['Date'].dt.normalize()
final_df = final_df.drop(columns=['Date'])
final_df['day_of_week'] = final_df['date'].dt.day_name()
final_df.columns = ['ticker', 'open', 'high', 'low', 'close', 'volume', 'date', 'day_of_week']
final_df

,ticker,open,high,low,close,volume,date,day_of_week
0,AAPL,26.6898,26.8154,26.4434,26.7722,120355375.0,2017-01-03,Tuesday
1,AAPL,26.7024,26.8576,26.6798,26.7428,89031733.0,2017-01-04,Wednesday
2,AAPL,26.7192,26.9361,26.6927,26.8812,94131019.0,2017-01-05,Thursday
3,AAPL,26.9186,27.2369,26.8479,27.1759,134161434.0,2017-01-06,Friday
4,AAPL,27.1846,27.5304,27.1818,27.4244,137261843.0,2017-01-09,Monday
...,...,...,...,...,...,...,...,...
106433,XOM,138.6400,145.0100,138.3100,143.7300,37577844.0,2026-02-03,Tuesday
106434,XOM,144.1150,147.8400,144.1150,147.5900,29426241.0,2026-02-04,Wednesday
106435,XOM,146.5800,146.7300,143.7950,146.0800,18431739.0,2026-02-05,Thursday
106436,XOM,146.6200,149.5700,146.0900,149.0500,17216025.0,2026-02-06,Friday


In [10]:
sp_500_index = pd.read_csv('sp500_index.csv')
sp_500_index.columns = ['date', 'S&P500']
sp_500_index['date'] = pd.to_datetime(sp_500_index['date'])
sp_500_index.head()

,date,S&P500
0,2014-12-22,2078.54
1,2014-12-23,2082.17
2,2014-12-24,2081.88
3,2014-12-26,2088.77
4,2014-12-29,2090.57


In [11]:
sp_500_companies = pd.read_csv('sp500_companies.csv')
sp_500_companies = sp_500_companies[['Symbol', 'Sector', 'Industry', 'Marketcap', 'Ebitda', 'Revenuegrowth', 'Fulltimeemployees', 'Longbusinesssummary', 'Weight']]
sp_500_companies.columns = ['ticker', 'sector', 'industry', 'marketcap', 'ebitda', 'revenue_growth', 'full_time_employees', 'long_business_summary', 'weight']
sp_500_companies.head()

,ticker,sector,industry,marketcap,ebitda,revenue_growth,full_time_employees,long_business_summary,weight
0,AAPL,Technology,Consumer Electronics,3846819807232,1.346610e+11,0.061,164000.0,"Apple Inc. designs, manufactures, and markets ...",0.069209
1,NVDA,Technology,Semiconductors,3298803056640,6.118400e+10,1.224,29600.0,NVIDIA Corporation provides graphics and compu...,0.059350
2,MSFT,Technology,Software - Infrastructure,3246068596736,1.365520e+11,0.160,228000.0,Microsoft Corporation develops and supports so...,0.058401
3,AMZN,Consumer Cyclical,Internet Retail,2365033807872,1.115830e+11,0.110,1551000.0,"Amazon.com, Inc. engages in the retail sale of...",0.042550
4,GOOGL,Communication Services,Internet Content & Information,2351625142272,1.234700e+11,0.151,181269.0,Alphabet Inc. offers various products and plat...,0.042309


In [12]:
sp_500_companies.shape

(502, 9)

In [13]:
sp_500_index.date.min(), sp_500_index.date.max()

(Timestamp('2014-12-22 00:00:00'), Timestamp('2024-12-20 00:00:00'))

In [14]:
df.date.min(), df.date.max()

(Timestamp('2017-01-04 00:00:00'), Timestamp('2023-12-27 00:00:00'))

In [15]:
df = df.merge(sp_500_index, on='date', how='left')
df = df.merge(sp_500_companies, on='ticker')
df.head()

,date,prev_date,future_date,title,description,maintext,ticker,prev_day_price,curr_day_price,next_day_price,...,title + description,S&P500,sector,industry,marketcap,ebitda,revenue_growth,full_time_employees,long_business_summary,weight
0,2019-01-22,2019-01-21,2019-01-23,UBS Warns on Client Activity After $13 Billion...,Withdrawals at the Zurich-based bank’s key glo...,(Bloomberg) -- UBS Group AG warned client acti...,C,63.12,61.85,62.13,...,UBS Warns on Client Activity After $13 Billion...,2632.90,Financial Services,Banks - Diversified,130856288256,NaN,-0.024,229000.0,"Citigroup Inc., a diversified financial servic...",0.002354
1,2019-10-31,2019-10-30,2019-11-01,Wayfair Plunges as Forecast Miss Heightens Gro...,(Bloomberg) -- Wayfair Inc. plunged near its l...,(Bloomberg) -- Wayfair Inc. plunged near its l...,C,72.97,71.86,73.84,...,Wayfair Plunges as Forecast Miss Heightens Gro...,3037.56,Financial Services,Banks - Diversified,130856288256,NaN,-0.024,229000.0,"Citigroup Inc., a diversified financial servic...",0.002354
2,2019-01-15,2019-01-14,2019-01-16,TSMC's $43 Billion Wipeout Will Only Get Worse...,TSMC will struggle to plug the hole left by Ap...,(Bloomberg) -- The worst isn’t over yet for Ta...,C,58.93,61.38,62.19,...,TSMC's $43 Billion Wipeout Will Only Get Worse...,2610.30,Financial Services,Banks - Diversified,130856288256,NaN,-0.024,229000.0,"Citigroup Inc., a diversified financial servic...",0.002354
3,2019-08-06,2019-08-05,2019-08-07,Canadian Stocks Are on Their Worst Streak in 1...,(Bloomberg) -- Canadian stocks played catch up...,(Bloomberg) -- Canadian stocks played catch up...,C,65.18,66.25,65.14,...,Canadian Stocks Are on Their Worst Streak in 1...,2881.77,Financial Services,Banks - Diversified,130856288256,NaN,-0.024,229000.0,"Citigroup Inc., a diversified financial servic...",0.002354
4,2019-02-08,2019-02-07,2019-02-09,Iron Ore Powers to Highest Since 2014 as Vale ...,Vale invoked force majeure earlier this week a...,(Bloomberg) -- Iron ore futures surged more th...,C,62.81,62.01,61.61,...,Iron Ore Powers to Highest Since 2014 as Vale ...,2707.88,Financial Services,Banks - Diversified,130856288256,NaN,-0.024,229000.0,"Citigroup Inc., a diversified financial servic...",0.002354


### Current-day OHLC and volume from final_df

Attach open, high, low, and volume for each (ticker, date) from the price dataset.

In [16]:
# Bring in current-day open/high/low/volume from final_df
curr_ohlcv = final_df[['ticker', 'date', 'open', 'high', 'low', 'volume']].rename(columns={
    'open':   'curr_open',
    'high':   'curr_high',
    'low':    'curr_low',
    'volume': 'curr_volume'
})
df = df.merge(curr_ohlcv, on=['ticker', 'date'], how='left')

# Intraday range and gap (close vs open)
df['curr_day_range']   = (df['curr_high'] - df['curr_low'])    / df['curr_low']        # high-low range as %
df['curr_day_gap']     = (df['curr_day_price'] - df['curr_open']) / df['curr_open']     # close vs open as %
df['curr_upper_wick']  = (df['curr_high'] - df['curr_day_price']) / df['curr_day_price']  # upper wick
df['curr_lower_wick']  = (df['curr_day_price'] - df['curr_low'])  / df['curr_day_price']  # lower wick

print(df[['curr_open','curr_high','curr_low','curr_volume','curr_day_range','curr_day_gap']].head())

   curr_open  curr_high  curr_low   curr_volume  curr_day_range  curr_day_gap
0    54.6417    54.7649   53.5609  2.822432e+07        0.022479      0.131919
1    64.4219    64.9662   63.5076  1.894149e+07        0.022967      0.115459
2    52.2178    53.9005   51.9232  4.896714e+07        0.038081      0.175461
3    58.7609    59.0545   57.7193  1.394045e+07        0.023133      0.127450
4    54.5516    54.8941   53.5697  1.796164e+07        0.024723      0.136722


### Lag features — price, volume, S&P500

For each lag N ∈ {2, 3, 4, 5, 7, 14, 21, 28} we look up the most recent trading-day
close/volume for each ticker on or before `date − N calendar days`.
The same backward-fill is applied to S&P 500.

`prev_day_price` already covers **lag-1** (one trading-day back), so we start at lag-2.

In [ ]:
LAGS = [2, 3, 4, 5, 7, 14, 21, 28]

# ── prepare lookups ──────────────────────────────────────────────────────────
# merge_asof requires the join key to be globally monotonically non-decreasing.
# Sort by date only (not ['ticker', 'date']) so the key column is globally sorted;
# the `by='ticker'` parameter handles per-ticker matching internally.
stock_prices = (
    final_df[['ticker', 'date', 'close', 'volume']]
    .sort_values('date')
    .reset_index(drop=True)
)

sp500_lookup = (
    sp_500_index[['date', 'S&P500']]
    .sort_values('date')
    .reset_index(drop=True)
)

# ── per-lag lookups ──────────────────────────────────────────────────────────
for lag in LAGS:
    # ── stock price & volume ─────────────────────────────────────────────────
    key_df = (
        df[['ticker', 'date']]
        .copy()
        .assign(orig_idx=df.index, lag_date=df['date'] - pd.Timedelta(days=lag))
        .sort_values('lag_date')   # sort by key only so merge_asof sees a globally sorted key
    )

    merged_price = pd.merge_asof(
        key_df,
        stock_prices.rename(columns={'date': 'price_date',
                                     'close': f'close_lag{lag}',
                                     'volume': f'volume_lag{lag}'}),
        left_on='lag_date',
        right_on='price_date',
        by='ticker',
        direction='backward'
    ).set_index('orig_idx')

    df[f'close_lag{lag}']  = merged_price[f'close_lag{lag}'].reindex(df.index)
    df[f'volume_lag{lag}'] = merged_price[f'volume_lag{lag}'].reindex(df.index)

    # ── S&P 500 (one lookup per unique date, then map back) ──────────────────
    unique_dates = (
        df[['date']].drop_duplicates()
        .assign(lag_date=lambda x: x['date'] - pd.Timedelta(days=lag))
        .sort_values('lag_date')
    )

    sp500_by_date = pd.merge_asof(
        unique_dates,
        sp500_lookup.rename(columns={'date': 'sp_date', 'S&P500': f'sp500_lag{lag}'}),
        left_on='lag_date',
        right_on='sp_date',
        direction='backward'
    )[['date', f'sp500_lag{lag}']]

    df = df.merge(sp500_by_date, on='date', how='left')

print('Lag columns added:', [c for c in df.columns if 'lag' in c])
df[[f'close_lag{l}' for l in LAGS] + [f'sp500_lag{l}' for l in LAGS]].head()

### Percentage changes and direction between lags and current price

In [ ]:
# Stock: return from lagN to current day close
for lag in LAGS:
    lag_close = df[f'close_lag{lag}']
    df[f'ret_lag{lag}']  = (df['curr_day_price'] - lag_close) / lag_close   # percent change
    df[f'dir_lag{lag}']  = (df[f'ret_lag{lag}'] > 0).astype(int)            # 1 = up, 0 = down/flat

# S&P 500: return from lagN to current day S&P500
for lag in LAGS:
    lag_sp = df[f'sp500_lag{lag}']
    df[f'sp500_ret_lag{lag}'] = (df['S&P500'] - lag_sp) / lag_sp
    df[f'sp500_dir_lag{lag}'] = (df[f'sp500_ret_lag{lag}'] > 0).astype(int)

df[[f'ret_lag{l}' for l in LAGS]].describe()

### Volume change features

In [ ]:
# Volume change: current day vs each lag (handles zero-volume edge cases)
for lag in LAGS:
    lag_vol = df[f'volume_lag{lag}'].replace(0, np.nan)
    df[f'vol_change_lag{lag}'] = (df['curr_volume'] - lag_vol) / lag_vol

# Average volume over medium / long windows
df['avg_vol_7d']  = df[['volume_lag2','volume_lag3','volume_lag4','volume_lag5','volume_lag7']].mean(axis=1)
df['avg_vol_28d'] = df[['volume_lag7','volume_lag14','volume_lag21','volume_lag28']].mean(axis=1)

# Relative volume: current day vs recent average
df['rel_vol_7d']  = df['curr_volume'] / df['avg_vol_7d'].replace(0, np.nan)
df['rel_vol_28d'] = df['curr_volume'] / df['avg_vol_28d'].replace(0, np.nan)

df[['curr_volume','avg_vol_7d','avg_vol_28d','rel_vol_7d','rel_vol_28d']].head()

### Alpha (stock return minus market return) per lag

In [ ]:
# Alpha: how much did the stock outperform (or underperform) S&P 500 over each window
for lag in LAGS:
    df[f'alpha_lag{lag}'] = df[f'ret_lag{lag}'] - df[f'sp500_ret_lag{lag}']

df[[f'alpha_lag{l}' for l in LAGS]].head()

### Momentum features

Differences between returns at various horizons capture acceleration / mean-reversion.

In [ ]:
# Short-term (2d) vs medium-term (7d)
df['momentum_2_7']  = df['ret_lag2']  - df['ret_lag7']
# Medium-term (7d) vs long-term (28d)
df['momentum_7_28'] = df['ret_lag7']  - df['ret_lag28']
# Short-term (2d) vs long-term (28d)
df['momentum_2_28'] = df['ret_lag2']  - df['ret_lag28']
# Acceleration: 2d momentum vs 5d momentum
df['momentum_2_5']  = df['ret_lag2']  - df['ret_lag5']
# Medium momentum: 5d vs 14d
df['momentum_5_14'] = df['ret_lag5']  - df['ret_lag14']

# Same for S&P 500
df['sp500_momentum_2_7']  = df['sp500_ret_lag2']  - df['sp500_ret_lag7']
df['sp500_momentum_7_28'] = df['sp500_ret_lag7']  - df['sp500_ret_lag28']
df['sp500_momentum_2_28'] = df['sp500_ret_lag2']  - df['sp500_ret_lag28']

df[['momentum_2_7','momentum_7_28','momentum_2_28','sp500_momentum_2_7']].head()

### Price position within recent range

Where does the current price sit relative to its recent high/low? Useful for mean-reversion signals.

In [ ]:
short_lag_cols = [f'close_lag{l}' for l in [2, 3, 4, 5, 7]]
long_lag_cols  = [f'close_lag{l}' for l in [7, 14, 21, 28]]

# High/low over ~7 calendar days
df['recent_high_7d'] = df[short_lag_cols].max(axis=1)
df['recent_low_7d']  = df[short_lag_cols].min(axis=1)

# High/low over ~28 calendar days
df['recent_high_28d'] = df[long_lag_cols].max(axis=1)
df['recent_low_28d']  = df[long_lag_cols].min(axis=1)

# Current price vs those levels
df['price_vs_7d_low']   = (df['curr_day_price'] - df['recent_low_7d'])   / df['recent_low_7d']
df['price_vs_7d_high']  = (df['curr_day_price'] - df['recent_high_7d'])  / df['recent_high_7d']
df['price_vs_28d_low']  = (df['curr_day_price'] - df['recent_low_28d'])  / df['recent_low_28d']
df['price_vs_28d_high'] = (df['curr_day_price'] - df['recent_high_28d']) / df['recent_high_28d']

# Normalised position within range [0, 1]  (0 = at low, 1 = at high)
range_7d  = (df['recent_high_7d']  - df['recent_low_7d']).replace(0, np.nan)
range_28d = (df['recent_high_28d'] - df['recent_low_28d']).replace(0, np.nan)
df['price_position_7d']  = (df['curr_day_price'] - df['recent_low_7d'])  / range_7d
df['price_position_28d'] = (df['curr_day_price'] - df['recent_low_28d']) / range_28d

df[['price_vs_7d_low','price_vs_7d_high','price_position_7d','price_position_28d']].head()

### Approximate historical volatility

Computed as the standard deviation of all available lag returns — a proxy for recent price dispersion.

In [ ]:
# Build a matrix of single-period returns between consecutive lag close prices
# ordered from oldest to newest:  lag28 → lag21 → lag14 → lag7 → lag5 → lag4 → lag3 → lag2 → lag1 (prev) → curr
lag_close_cols_ordered = (
    ['close_lag28', 'close_lag21', 'close_lag14', 'close_lag7',
     'close_lag5', 'close_lag4', 'close_lag3', 'close_lag2',
     'prev_day_price', 'curr_day_price']
)

price_matrix = df[lag_close_cols_ordered].values.astype(float)

# log returns between consecutive available prices
log_ret_matrix = np.where(
    (price_matrix[:, :-1] > 0) & (price_matrix[:, 1:] > 0),
    np.log(price_matrix[:, 1:] / price_matrix[:, :-1]),
    np.nan
)

df['hist_volatility_approx'] = np.nanstd(log_ret_matrix, axis=1)

# Short-window version (last ~7 days)
short_price_matrix = df[['close_lag7','close_lag5','close_lag4',
                          'close_lag3','close_lag2','prev_day_price','curr_day_price']].values.astype(float)
short_log_ret = np.where(
    (short_price_matrix[:, :-1] > 0) & (short_price_matrix[:, 1:] > 0),
    np.log(short_price_matrix[:, 1:] / short_price_matrix[:, :-1]),
    np.nan
)
df['hist_volatility_7d'] = np.nanstd(short_log_ret, axis=1)

df[['hist_volatility_approx','hist_volatility_7d']].describe()

### Calendar features

In [ ]:
df['month']           = df['date'].dt.month
df['quarter']         = df['date'].dt.quarter
df['week_of_year']    = df['date'].dt.isocalendar().week.astype(int)
df['is_month_start']  = df['date'].dt.is_month_start.astype(int)
df['is_month_end']    = df['date'].dt.is_month_end.astype(int)
df['is_quarter_end']  = df['date'].dt.is_quarter_end.astype(int)

# Numeric day-of-week (0=Mon … 4=Fri) — already have string version from v2
df['day_of_week_num'] = df['date'].dt.dayofweek

df[['month','quarter','week_of_year','is_month_start','is_month_end','is_quarter_end','day_of_week_num']].head()

### Quick sanity check on new columns

In [ ]:
new_cols = (
    [f'close_lag{l}'      for l in LAGS] +
    [f'volume_lag{l}'     for l in LAGS] +
    [f'sp500_lag{l}'      for l in LAGS] +
    [f'ret_lag{l}'        for l in LAGS] +
    [f'dir_lag{l}'        for l in LAGS] +
    [f'sp500_ret_lag{l}'  for l in LAGS] +
    [f'sp500_dir_lag{l}'  for l in LAGS] +
    [f'vol_change_lag{l}' for l in LAGS] +
    [f'alpha_lag{l}'      for l in LAGS]
)
print(f"Total columns in df: {len(df.columns)}")
print(f"New lag-based columns: {len(new_cols)}")
print(f"\nNull rates for lag close prices:")
print(df[[f'close_lag{l}' for l in LAGS]].isnull().mean().round(3))

In [ ]:
df.head(3)

In [ ]:
df.columns.tolist()

In [ ]:
df.to_parquet('data_sample/news_prices_full_processed_with_target_v2.parquet', index=False)
print('Saved:', df.shape)

### Notes and TODOs

**New features added in v3:**

| Group | Columns |
|---|---|
| Current-day OHLCV | `curr_open`, `curr_high`, `curr_low`, `curr_volume`, `curr_day_range`, `curr_day_gap`, `curr_upper_wick`, `curr_lower_wick` |
| Lag close prices | `close_lag{2,3,4,5,7,14,21,28}` |
| Lag volumes | `volume_lag{2,3,4,5,7,14,21,28}` |
| Lag S&P 500 | `sp500_lag{2,3,4,5,7,14,21,28}` |
| Stock returns to current | `ret_lag{N}`, `dir_lag{N}` |
| S&P 500 returns to current | `sp500_ret_lag{N}`, `sp500_dir_lag{N}` |
| Volume changes | `vol_change_lag{N}`, `avg_vol_7d`, `avg_vol_28d`, `rel_vol_7d`, `rel_vol_28d` |
| Alpha (stock − market) | `alpha_lag{N}` |
| Momentum | `momentum_2_7`, `momentum_7_28`, `momentum_2_28`, `momentum_2_5`, `momentum_5_14`, `sp500_momentum_*` |
| Price range position | `recent_high/low_7d/28d`, `price_vs_*`, `price_position_7d/28d` |
| Historical volatility | `hist_volatility_approx`, `hist_volatility_7d` |
| Calendar | `month`, `quarter`, `week_of_year`, `is_month_start`, `is_month_end`, `is_quarter_end`, `day_of_week_num` |

**Notes:**

- Lag dates are calendar-day offsets; prices are backward-filled to the nearest available trading day.
- Volume and price features should be treated as **look-back only** — they reflect information available *before* the prediction date.
- `next_curr_per_change` (and `next_curr_pos_change`) remain the primary targets.

**TODO:**

- Remove outliers from `next_curr_per_change` (e.g., rows where `|next_curr_per_change| > 0.25`).
- Consider adding risk-free rate features (T-bill yield).
- SHAP analysis to evaluate feature importances after modeling.